# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
 
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed (use --upgrade to get the latest version)
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object (not a dict)

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Citation: {getattr(metadata, 'citeAs', None)}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their fields
from pprint import pprint

# Retrieve record sets (by @id)
record_sets = []
if hasattr(metadata, 'recordSet'):
    for record_set in metadata.recordSet:
        record_sets.append(record_set['@id'] if isinstance(record_set, dict) else record_set)
else:
    print('No record sets found in the dataset metadata.')

print('Record sets in dataset:')
pprint(record_sets)

# For each record set, list its fields and columns by @id
if record_sets:
    for record_set_id in record_sets:
        print(f"\nFields for record set @id: {record_set_id}")
        record_set_obj = [rs for rs in metadata.recordSet if (rs['@id'] if isinstance(rs, dict) else rs) == record_set_id][0]
        if isinstance(record_set_obj, dict) and 'field' in record_set_obj:
            for field in record_set_obj['field']:
                if isinstance(field, dict):
                    field_id = field.get('@id')
                    column = field.get('column', None)
                else:
                    field_id = field
                    column = None
                print(f"  - Field @id: {field_id}")
                if column:
                    if isinstance(column, dict):
                        print(f"    - Column @id: {column.get('@id', column)}")
                    elif isinstance(column, list):
                        print("    - Columns:")
                        for c in column:
                            if isinstance(c, dict):
                                print(f"      - Column @id: {c.get('@id', c)}")
                            else:
                                print(f"      - Column @id: {c}")
                    else:
                        print(f"    - Column @id: {column}")
        else:
            print('  No fields found in this record set.')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. 

**Note:** All references (`record_set`, `field`, `column`) use their `@id`.

In [ ]:
# If record sets exist, extract all into DataFrames by @id
dataframes = {}
for record_set_id in record_sets:
    print(f"Extracting data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  -> Columns: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalization, and grouping. 

*- Update `<numeric_field_id>` and `<group_field_id>` to match relevant column `@id`s printed in previous steps for your main record set.*

In [ ]:
# --- Replace with actual record set @id (example below uses a hypothetical @id) ---
main_record_set_id = record_sets[0] if record_sets else None
df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()

# Display summary and choose a numeric field for analysis
print('Numeric columns:')
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(numeric_cols)

if numeric_cols:
    numeric_field_id = numeric_cols[0]
    threshold = df[numeric_field_id].quantile(0.90) if len(df) > 0 else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (pick first string column)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field_id = cat_cols[0] if cat_cols else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print('No suitable group field found.')
else:
    print('No numeric columns available for EDA in this record set.')

## 5. Visualization
Visualize distributions or relationships between fields in the record set. 

*- Edit to choose relevant fields and plots for your analysis as needed.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by category if available
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we:

- Loaded rich and structured dataset metadata via its Croissant schema (`@id`-first referencing).
- Explored available record sets and reviewed field and column structure.
- Loaded record set data, performed basic EDA (including filtering, normalization, grouping), and visualized results.

This workflow can be extended to more advanced analysis using the `mlcroissant` library and your own domain expertise.